In [ ]:
import pyreadstat
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path().resolve().parent  # this should be R-learner-LTE
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Replace with your actual path
sas_path = r"C:\Users\ma\Research\Long-term-effects\IST-3-Dataset\datashare_aug2015.sas7bdat"

df, meta = pyreadstat.read_sas7bdat(sas_path)

#print(df.shape)
#print(df.head())
df["A"] = (df["itt_treat"] == 0).astype(int)
X_cols = [
    "age", "gender", "randdelay", "country",
    "livealone_rand", "indepinadl_rand",
    "sbprand", "dbprand", "weight", "glucose",
    "gcs_score_rand", "nihss",
    "atrialfib_rand", "stroke_pre",
    "hypertension_pre", "diabetes_pre",
    "aspirin_pre", "other_antiplat_pre", "anticoag_pre",
    "stroketype",
    "R_infarct_size", "R_hypodensity", "R_swelling"
]




S_cols = ["sich7", "dead7", "indepinadl_7", "ablewalk_7",
          "gcs_eye_7", "gcs_motor_7", "gcs_verbal_7"]

Y_col = "ohs6"
type(df)
A_col = "A"
kee_cols = X_cols + S_cols + [Y_col, A_col]
df = df[kee_cols].dropna().reset_index(drop=True)
print(df.shape)
df.head()

save_path = r"C:\Users\ma\Research\Long-term-effects\R-learner-LTE\src\data\semi_synthetic_real_data.csv"
df.to_csv(save_path, index=False)

(2720, 32)


: 

In [ ]:
assert len(meta.column_names) == len(meta.column_labels)
for name, label in zip(meta.column_names, meta.column_labels):
    print(f"{name}: {label}")

randhosp_id: Hosp_ID (anonymised)
randpat_id: Patient_ID (anonymised)
pretrialexp: Centre with pre-trial experience of thrombolysis
country: Country
trialphase: Trial phase
phase: None
itt_treat: Allocated treatment
age: Age at randomisation
gender: Gender
deathcode: Cause of death (IST3 E codes)
deathdate_unknown: None
randyear: Year of randomisation
randmonth: Month of randomisation
randhour: Local hour of randomisation
randmin: Local minute of randomisation
randdelay: Delay (hours) from stroke to randomisation
livealone_rand: Lived alone before stroke?
indepinadl_rand: Independent in ADL before stroke?
nobleed_rand: Imaging excluded intracranial haemorrhage?
infarct: Recent ischaemic change likely cause of this stroke?
antiplat_rand: Received antiplatelet drugs in last 48 hours?
atrialfib_rand: Patient in atrial fibrillation at randomisation?
sbprand: Systolic BP at randomisation (mm Hg)
dbprand: Diastolic BP at randomisation (mm Hg)
weight: Estimated weight (kg)
glucose: Blood gluc

In [35]:
cat_cols = [
    "gender", "country", "stroketype",
    "livealone_rand", "indepinadl_rand",
    "atrialfib_rand", "stroke_pre", "hypertension_pre", "diabetes_pre",
    "aspirin_pre", "other_antiplat_pre", "anticoag_pre",
    "sich7", "dead7", "indepinadl_7", "ablewalk_7",
    "gcs_eye_7", "gcs_motor_7", "gcs_verbal_7",
]
bad_vals = {20, 30, 40}
tmp = df[cat_cols].apply(pd.to_numeric, errors="coerce")

bad_mask = tmp.isin(list(bad_vals)).any(axis=1)

print(f"Rows before: {len(df)}")
df_clean = df.loc[~bad_mask].copy()
print(f"Rows after : {len(df_clean)}")
print(f"Dropped    : {bad_mask.sum()}")

df = df_clean.reset_index(drop=True)
gcs_cols = ["gcs_eye_7", "gcs_motor_7", "gcs_verbal_7"]
df[gcs_cols] = df[gcs_cols].apply(pd.to_numeric, errors="coerce")
df[gcs_cols] = df[gcs_cols].replace({10: 0}) #map death code to numeric 0
df = df.dropna(subset=gcs_cols).copy()

Rows before: 2720
Rows after : 2664
Dropped    : 56


In [36]:
#separate numeric and categorical columns of X
X_numeric_cols = [
    "age", "randdelay", "sbprand", "dbprand", "weight", "glucose",
    "gcs_score_rand", "nihss",
    "R_infarct_size","R_hypodensity","R_swelling"
]
X_categorical_cols = [col for col in X_cols if col not in X_numeric_cols]

S_numeric_cols = [
    "gcs_eye_7", "gcs_motor_7", "gcs_verbal_7"
]
S_categorical_cols = [col for col in S_cols if col not in S_numeric_cols]
print("X_categorical_cols:", X_categorical_cols)
print("S_categorical_cols:", S_categorical_cols)
#print(df[:10][X_numeric_cols])
#print(df[:15][X_categorical_cols])
#Convert categorical columns to category dtype
for col in X_categorical_cols:
    df[col] = df[col].astype('category')
for col in S_categorical_cols:
    df[col] = df[col].astype('category')

for col in df.columns:
    #show data range for the column
    if col in X_numeric_cols + S_numeric_cols:
        print(f"Numeric {col}: {df[col].dtype}, min: {df[col].min()}, max: {df[col].max()}")
    elif col in X_categorical_cols + S_categorical_cols:
        print(f"Categorical {col}: {df[col].dtype}, categories: {df[col].cat.categories.tolist()}")
    else:
        print(f"{col}: {df[col].dtype}, unique values: {df[col].nunique()}")

X_categorical_cols: ['gender', 'country', 'livealone_rand', 'indepinadl_rand', 'atrialfib_rand', 'stroke_pre', 'hypertension_pre', 'diabetes_pre', 'aspirin_pre', 'other_antiplat_pre', 'anticoag_pre', 'stroketype']
S_categorical_cols: ['sich7', 'dead7', 'indepinadl_7', 'ablewalk_7']
Numeric age: float64, min: 31.935, max: 97.846
Categorical gender: category, categories: [1.0, 2.0]
Numeric randdelay: float64, min: 0.1, max: 24.0
Categorical country: category, categories: ['AUSTRALIA', 'BELGIUM', 'ITALY', 'NORWAY', 'OTHER', 'POLAND', 'PORTUGAL', 'SWEDEN', 'UK']
Categorical livealone_rand: category, categories: [1.0, 2.0]
Categorical indepinadl_rand: category, categories: [1.0, 2.0]
Numeric sbprand: float64, min: 90.0, max: 220.0
Numeric dbprand: float64, min: 35.0, max: 130.0
Numeric weight: float64, min: 31.0, max: 150.0
Numeric glucose: float64, min: 3.0, max: 20.0
Numeric gcs_score_rand: float64, min: 3.0, max: 15.0
Numeric nihss: float64, min: 0.0, max: 37.0
Categorical atrialfib_rand

In [ ]:
# Use xgboostto estimate \E[S \mid A=1, X] and \E[S \mid A=0, X]
import xgboost as xgb
from sklearn.model_selection import train_test_split
import numpy as np
X = df[X_cols]
A = df[A_col]
S = df[S_cols]
#normalize X and S
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)
scaler_S = StandardScaler()
S = scaler_S.fit_transform(S)

print(S[:10])
print(X[:10])

In [37]:
df.columns

Index(['age', 'gender', 'randdelay', 'country', 'livealone_rand',
       'indepinadl_rand', 'sbprand', 'dbprand', 'weight', 'glucose',
       'gcs_score_rand', 'nihss', 'atrialfib_rand', 'stroke_pre',
       'hypertension_pre', 'diabetes_pre', 'aspirin_pre', 'other_antiplat_pre',
       'anticoag_pre', 'stroketype', 'R_infarct_size', 'R_hypodensity',
       'R_swelling', 'sich7', 'dead7', 'indepinadl_7', 'ablewalk_7',
       'gcs_eye_7', 'gcs_motor_7', 'gcs_verbal_7', 'ohs6', 'A'],
      dtype='object')

In [ ]:
from src.model.s_conditional import ConditionalSModeler

modeler = ConditionalSModeler(
    a_col="A",
    x_numeric_cols=X_numeric_cols,
    x_categorical_cols=X_categorical_cols,
    s_numeric_cols=S_numeric_cols,          # e.g., ['nihss_7', ...]
    s_categorical_cols=S_categorical_cols,  # e.g., ['sich7', 'dead7', ...]
    test_size=0.2,
    random_state=42,
    reg_params={"n_estimators": 100},       # optional overrides
    clf_params={"max_depth": 5},
)
models = modeler.fit(df, s_cobls=S_cols)      # S_cols = S_numeric_cols + S_categorical_cols
print(modeler.metrics())                  # {('dead7',0): ('accuracy',0.83), ...}
mu_s1_dead7 = modeler.predict_expectation(df[X_numeric_cols + X_categorical_cols], s_col="dead7", a_val=1)
#mu_s0_nihss = modeler.predict_expectation(df[X_numeric_cols + X_categorical_cols], s_col="nihss_7", a_val=0)

{('sich7', 0): ('accuracy', 0.9887218045112782), ('sich7', 1): ('accuracy', 0.9400749063670412), ('dead7', 0): ('accuracy', 0.9360902255639098), ('dead7', 1): ('accuracy', 0.8951310861423221), ('indepinadl_7', 0): ('accuracy', 0.7218045112781954), ('indepinadl_7', 1): ('accuracy', 0.6441947565543071), ('ablewalk_7', 0): ('accuracy', 0.7218045112781954), ('ablewalk_7', 1): ('accuracy', 0.6104868913857678), ('gcs_eye_7', 0): ('mse', 1.1524232124467815), ('gcs_eye_7', 1): ('mse', 1.1985285363174216), ('gcs_motor_7', 0): ('mse', 2.5838122241665644), ('gcs_motor_7', 1): ('mse', 2.8120444107766813), ('gcs_verbal_7', 0): ('mse', 1.7584416931500753), ('gcs_verbal_7', 1): ('mse', 1.8754482631711886)}


KeyError: "No model fitted for ('nihss_7', 0)"

In [39]:
S_numeric_cols

['gcs_eye_7', 'gcs_motor_7', 'gcs_verbal_7']

In [ ]:

train, test = train_test_split(df, test_size=0.2, random_state=42)
X_train, X_test = train[X_cols], test[X_cols]
A_train, A_test = train[A_col], test[A_col]
S_train, S_test = train[S_cols], test[S_cols]

models_treated = {}
models_untreated = {}
for s_col in S_cols:
    #train model for each S_col and each treatment group
    treated_idx = A_train == 1
    untreated_idx = A_train == 0
    #assert that sum of treated_idx and untreated_idx equals length of A_train
    assert np.sum(treated_idx) + np.sum(untreated_idx) == len(A_train)
    models_treated[s_col] = xgb.XGBRegressor(n_estimators=100, random_state=42)
    models_treated[s_col].fit(X_train[treated_idx], S_train.loc[treated_idx, s_col])
    models_untreated[s_col] = xgb.XGBRegressor(n_estimators=100, random_state=42)
    models_untreated[s_col].fit(X_train[untreated_idx], S_train.loc[untreated_idx, s_col])

    #test the models and report accuracy
    treated_test_idx = A_test == 1
    untreated_test_idx = A_test == 0
    assert np.sum(treated_test_idx) + np.sum(untreated_test_idx) == len(A_test)
    treated_preds = models_treated[s_col].predict(X_test[treated_test_idx])
    untreated_preds = models_untreated[s_col].predict(X_test[untreated_test_idx])
    treated_true = S_test.loc[treated_test_idx, s_col]
    untreated_true = S_test.loc[untreated_test_idx, s_col]
    treated_mse = np.mean((treated_preds - treated_true) ** 2)
    untreated_mse = np.mean((untreated_preds - untreated_true) ** 2)